# Week 3 — live coding (completed)

**Social and Cultural Analytics (7AAVBCS5)**

The filled-in version of what we typed in the lecture, released afterwards so you can check what
you missed.

## Setup

In [1]:
import subprocess
from pathlib import Path

import pandas as pd

# The module's data lives in a separate public repository so that this notebook
# works both on your own machine and on Colab.
DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a file or folder in the module's data folder.

    A notebook's "current folder" is wherever it was opened from, which differs
    between Positron and Colab and is the usual cause of FileNotFoundError. So
    rather than assuming, we search upwards for a `data` folder. If there isn't
    one -- which means we are on Colab -- the public data repository is
    downloaded once and used instead.

    Works for single files ("museum_data.csv") and for folders ("SUA").
    """
    here = Path.cwd()
    for candidate in (here, *here.parents):
        local = candidate / "data" / name
        if local.exists():
            return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    target = downloaded / name
    if not target.exists():
        raise FileNotFoundError(f"{name!r} is not in the module data.")
    return target


print("we are running in:", Path.cwd())

we are running in: C:\Users\markj\OneDrive - King's College London\coding\GitHub\Teaching\7AAVBCS5\weeks\week-03


## 1. Creating a data frame

In [2]:
shows = pd.DataFrame({
    "title": ["Fleabag", "Arcane", "Chernobyl", "Bluey"],
    "year": [2016, 2021, 2019, 2018],
    "rating": [8.7, 9.0, 9.3, 9.4],
    "streaming_platform": ["iPlayer", "Netflix", "NOW", "iPlayer"],
    "watched_again": [True, True, False, True],
})
shows

,title,year,rating,streaming_platform,watched_again
0,Fleabag,2016,8.7,iPlayer,True
1,Arcane,2021,9.0,Netflix,True
2,Chernobyl,2019,9.3,NOW,False
3,Bluey,2018,9.4,iPlayer,True


A dictionary of columns: key is the column name, value is the list of entries.

In [3]:
shows.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               4 non-null      str    
 1   year                4 non-null      int64  
 2   rating              4 non-null      float64
 3   streaming_platform  4 non-null      str    
 4   watched_again       4 non-null      bool   
dtypes: bool(1), float64(1), int64(1), str(2)
memory usage: 315.0 bytes


In [4]:
shows.describe()

,year,rating
count,4.000000,4.000000
mean,2018.500000,9.100000
std,2.081666,0.316228
min,2016.000000,8.700000
25%,2017.500000,8.925000
50%,2018.500000,9.150000
75%,2019.500000,9.325000
max,2021.000000,9.400000


Note `describe()` only summarised the numeric columns — `year` and `rating`. R's `summary()`
described everything. For that here, use `shows.describe(include="all")`.

In [5]:
shows["years_old"] = 2026 - shows["year"]
shows

,title,year,rating,streaming_platform,watched_again,years_old
0,Fleabag,2016,8.7,iPlayer,True,10
1,Arcane,2021,9.0,Netflix,True,5
2,Chernobyl,2019,9.3,NOW,False,7
3,Bluey,2018,9.4,iPlayer,True,8


In [6]:
shows[shows["rating"] > 9]

,title,year,rating,streaming_platform,watched_again,years_old
2,Chernobyl,2019,9.3,NOW,False,7
3,Bluey,2018,9.4,iPlayer,True,8


## 2. Exploring the museum data

In [7]:
museum = pd.read_csv(data_path("museum_data.csv"))
print("shape:", museum.shape)
print("columns:", list(museum.columns))
museum.head()

shape: (150, 8)
columns: ['object_id', 'object_type', 'region', 'acquisition_date', 'acquisition_method', 'original_owner', 'cultural_significance', 'valuation']


,object_id,object_type,region,acquisition_date,acquisition_method,original_owner,cultural_significance,valuation
0,1,Pottery,Pacific Islands,1952,Purchase,NaN,Ceremonial,27000.0
1,2,Jewelry,Southeast Asia,2008,Colonial Collection,NaN,NaN,49000.0
2,3,Sculpture,Southeast Asia,1989,Purchase,NaN,Ceremonial,13000.0
3,4,Sculpture,West Africa,1917,Colonial Collection,NaN,NaN,10000.0
4,5,Textile,Pacific Islands,1923,NaN,NaN,Sacred,35000.0


### The overview table

A summary is itself a data frame. Build it the same way as any other — a dictionary of columns —
except the values come from methods on the original table.

In [8]:
overview = pd.DataFrame({
    "type": museum.dtypes.astype(str),
    "present": museum.notna().sum(),
    "missing": museum.isna().sum(),
    "missing_%": (museum.isna().mean() * 100).round(1),
    "distinct": museum.nunique(),
})
overview

,type,present,missing,missing_%,distinct
object_id,int64,150,0,0.0,150
object_type,str,150,0,0.0,5
region,str,150,0,0.0,5
acquisition_date,int64,150,0,0.0,91
acquisition_method,str,107,43,28.7,3
original_owner,str,16,134,89.3,2
cultural_significance,str,75,75,50.0,3
valuation,float64,146,4,2.7,48


Read that carefully, because it is the whole point of the exercise:

- `original_owner` is present for **16 of 150** objects — 89.3% missing — and has only **two**
  distinct values.
- `cultural_significance` is missing for exactly half.
- `acquisition_method` is missing for 43.

We come back to what that means at the end.

In [9]:
# What are those two values in original_owner?
print(museum["original_owner"].value_counts(dropna=False))

original_owner
NaN                 134
Known Community       9
Named Individual      7
Name: count, dtype: int64


So the column does not record *who* previously owned an object. It records whether the previous
owner was **identifiable at all** — and for 134 objects, not even that was written down.

### The most valuable object

In [10]:
museum.loc[museum["valuation"].idxmax()]

object_id                          81
object_type                   Jewelry
region                   North Africa
acquisition_date                 1928
acquisition_method           Donation
original_owner                    NaN
cultural_significance             NaN
valuation                     50000.0
Name: 80, dtype: object

### The oldest acquisition

In [11]:
museum.loc[museum["acquisition_date"].idxmin()]

object_id                                 94
object_type                          Jewelry
region                           West Africa
acquisition_date                        1880
acquisition_method       Colonial Collection
original_owner                           NaN
cultural_significance                    NaN
valuation                            26000.0
Name: 93, dtype: object

### Objects from one region

In [12]:
west_africa = museum[museum["region"] == "West Africa"]
print(west_africa.shape[0], "objects from West Africa")
west_africa.head(3)

31 objects from West Africa


,object_id,object_type,region,acquisition_date,acquisition_method,original_owner,cultural_significance,valuation
3,4,Sculpture,West Africa,1917,Colonial Collection,NaN,NaN,10000.0
13,14,Sculpture,West Africa,1912,NaN,NaN,Ceremonial,29000.0
18,19,Mask,West Africa,1969,Purchase,NaN,Domestic,36000.0


### Making a category

In [13]:
# Three bands, using the quartiles we saw in describe().
museum["value_category"] = pd.cut(
    museum["valuation"],
    bins=[0, 15000, 40000, 100000],
    labels=["low", "medium", "high"],
)
print(museum["value_category"].value_counts())

value_category
medium    73
low       38
high      35
Name: count, dtype: int64


`pd.cut` turns a number into a band. We use it properly in Week 5; for now just note that a
category is something you *construct*, not something the data hands you. Where you put the
boundaries changes what the analysis says.

## 3. `.loc` and `.iloc` — where it matters

In [14]:
# On the full table they agree, because the labels happen to be 0, 1, 2...
print("loc :", museum.loc[0, "object_type"])
print("iloc:", museum.iloc[0, 1])

loc : Pottery
iloc: Pottery


In [15]:
# Now filter, and look at what happened to the labels.
se_asia = museum[museum["region"] == "Southeast Asia"]
print("first five labels:", list(se_asia.index[:5]))
print("iloc[0] object_type:", se_asia.iloc[0]["object_type"])

first five labels: [1, 2, 8, 17, 24]
iloc[0] object_type: Jewelry


The labels came with the rows — they were **not** renumbered. So:

- `.iloc[0]` is "the first row of this table", which is what we usually mean.
- `.loc[0]` is "the row labelled 0", which may not be in this table at all — or worse, may be
  present but somewhere in the middle, giving a wrong answer with no error.

In [16]:
# If you want the labels renumbered after filtering:
se_asia_clean = se_asia.reset_index(drop=True)
print("after reset_index:", list(se_asia_clean.index[:5]))

after reset_index: [0, 1, 2, 3, 4]


## 4. Files

In [17]:
print("current folder:", Path.cwd())
print("is there a data folder above us?", (Path.cwd().parent.parent / "data").exists())

current folder: C:\Users\markj\OneDrive - King's College London\coding\GitHub\Teaching\7AAVBCS5\weeks\week-03
is there a data folder above us? True


In [18]:
fruit = pd.read_csv(data_path("fruitData.csv"))
fruit

,Fruit,Color,Shape,Juice
0,apple,red,round,1.0
1,banana,yellow,oblong,0.0
2,pear,green,pear,0.5
3,orange,orange,round,1.0
4,kiwi,green,round,0.0


In [19]:
print(fruit["Color"].value_counts())
print()
print(fruit["Shape"].value_counts())

Color
green     2
red       1
yellow    1
orange    1
Name: count, dtype: int64

Shape
round     3
oblong    1
pear      1
Name: count, dtype: int64


In [20]:
# Add a column, take a subset, save it.
fruit["is_round"] = fruit["Shape"] == "round"
round_fruit = fruit[fruit["is_round"]]
round_fruit.to_csv("round_fruit.csv", index=False)
print(round_fruit)

    Fruit   Color  Shape  Juice  is_round
0   apple     red  round    1.0      True
3  orange  orange  round    1.0      True
4    kiwi   green  round    0.0      True


In [21]:
# Read it back and check the columns are what we expect.
check = pd.read_csv("round_fruit.csv")
print(list(check.columns))

['Fruit', 'Color', 'Shape', 'Juice', 'is_round']


No `Unnamed: 0`, because we passed `index=False`. Compare with a file that was saved *without*
that argument:

In [22]:
bananas = pd.read_csv(data_path("bananas_apples.csv"))
print(list(bananas.columns))

['Unnamed: 0', 'Fruit', 'Color']


`Unnamed: 0` is a row-number column that got written into the file — in this case from R, without
`row.names = FALSE`. Always pass `index=False` when you save.

## 5. Debugging with AI

Two errors on purpose. Read each one before fixing it.

In [23]:
# Wrong capitalisation. Uncomment to see the KeyError.
# museum["Region"]

`KeyError: 'Region'` — there is no such column. Names are case sensitive; it is `region`.

If you paste this into <https://ai.create.kcl.ac.uk/chat>, include what you were trying to do, the
code, and the full error. Ask it to **explain** the error rather than rewrite your code — you will
need to recognise it again next week, in a closed quiz.

In [24]:
# `and` instead of `&`. Uncomment to see the ValueError.
# museum[(museum["region"] == "West Africa") and (museum["valuation"] > 20000)]

In [25]:
# The version that works.
museum[(museum["region"] == "West Africa") & (museum["valuation"] > 20000)].shape

(20, 9)

## What the overview table was telling us

Every region in this collection is in the Global South. `acquisition_method` reads
`Colonial Collection` for 29 objects, with 43 more unrecorded. The oldest acquisition, from 1880,
is a West African object with no provenance and no cultural significance recorded.

So the missing 89% in `original_owner` is not a data-quality problem to be cleaned up. It is a
record of how this collection was assembled, and of what the people assembling it did not think
worth writing down.

**That** is what "the dataset and its boundaries" means, and it is the second section of your data
overview.

End of live coding.